# Week 21 · Notebook 1: Unity Catalog & Delta Lake Lab

# Requirements: Databricks workspace (free trial) + upload the week-01 CSVs to a volume

Upload the three files below into the volume `/Volumes/zrl_/zorologistics/raw/`
(Catalog Explorer → `zrl_` → `zorologistics` → `raw` → "Upload to this volume"):

- `shipments.csv`
- `carriers.csv`
- `lanes.csv`

They are produced by `zoro/data.py` at the repo root. No secrets are used, run in your own trial workspace.


## What this notebook builds

1. A governed **Unity Catalog** namespace: catalog `zrl_` → schema `zorologistics` → volumes `raw` and `checkpoints`.
2. A **Delta table** loaded from a CSV in the volume (both the SQL `read_files` path and the PySpark path).
3. **Delta time travel** (`DESCRIBE HISTORY`, `SELECT … VERSION AS OF`), then **VACUUM**, **OPTIMIZE ZORDER**, and **liquid clustering**.

See the concepts in [`reference/knowledge-base/research/06-databricks-deep-dive.md`](../../knowledge-base/research/06-databricks-deep-dive.md) §1 and §3, and module files `00`, `04` in `reference/platforms/databricks/`.


In [ ]:
# Verify the runtime and identity. Everything below is governed by Unity Catalog.
print("Spark version:", spark.version)
print("Current user:", spark.sql("SELECT current_user()").collect()[0][0])
print("Default catalog:", spark.catalog.currentCatalog())
print("Default schema:", spark.catalog.currentDatabase())


## 1. Create the Unity Catalog namespace

Unity Catalog uses a **three-level namespace**: `catalog.schema.object`. A **catalog** isolates and organizes data; a **schema** (database) holds tables/views/volumes/functions/models; a **volume** governs *non-tabular* files and is addressed by the path `/Volumes/<catalog>/<schema>/<volume>/…`. We create all three idempotently. If your trial does not allow creating catalogs, reuse an existing catalog and adjust the variable below.


In [ ]:
catalog = "zrl_"       # note the trailing underscore
schema = "zorologistics"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.raw")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.checkpoints")

print(f"Ready: {catalog}.{schema} with volumes raw, checkpoints")


In [ ]:
# List the uploaded files so we know what we are loading.
volume_path = f"/Volumes/{catalog}/{schema}/raw"
files = dbutils.fs.ls(volume_path)
for f in files:
    print(f.name, "->", f.size, "bytes")
print("files in volume:", len(files))


## 2. Load a CSV into a Delta table

**Delta Lake is the default storage layer**: unless configured otherwise, every table is a Delta table. We load the raw shipments in two ways: `read_files` in SQL (below) and `spark.read` in Python (next). Bronze keeps source fidelity, so we do not clean yet.


In [ ]:
%sql
-- Load raw shipments into a managed Delta table. read_files reads directly from the volume.
CREATE OR REPLACE TABLE zrl_.zorologistics.shipments_bronze AS
SELECT *
FROM read_files('/Volumes/zrl_/zorologistics/raw/shipments.csv',
                format => 'csv',
                header => true,
                inferSchema => true)


In [ ]:
%sql
-- Sanity check: row count and a peek at the schema as inferred.
SELECT count(*) AS row_count FROM zrl_.zorologistics.shipments_bronze


In [ ]:
%sql
-- A five-row peek (bronze = source fidelity, so timestamps may still be strings).
SELECT * FROM zrl_.zorologistics.shipments_bronze LIMIT 5


In [ ]:
# Same load, but with PySpark, for the two dimension tables.
for name in ["carriers", "lanes"]:
    df = (spark.read.format("csv")
          .option("header", True)
          .option("inferSchema", True)
          .load(f"{volume_path}/{name}.csv"))
    df.write.mode("overwrite").saveAsTable(f"zrl_.zorologistics.{name}")
    print(name, "->", df.count(), "rows")


## 3. Delta time travel

Every write appends a new entry to Delta's transaction log, so a table is a *sequence of versions*. `DESCRIBE HISTORY` lists them; `SELECT … VERSION AS OF n` (or `TIMESTAMP AS OF …`) reads a prior state; `@`-syntax (`table@v1`) is a shorthand. To make the effect visible we build a small isolated demo table so we do not pollute bronze.


In [ ]:
%sql
-- Inspect the transaction log of the bronze table (one CREATE = version 0).
DESCRIBE HISTORY zrl_.zorologistics.shipments_bronze


In [ ]:
%sql
-- A small isolated demo table: version 1 holds 1000 rows.
CREATE OR REPLACE TABLE zrl_.zorologistics.time_travel_demo AS
SELECT * FROM zrl_.zorologistics.shipments_bronze LIMIT 1000


In [ ]:
%sql
-- Version 2: insert another 1000 rows, so the two versions differ.
INSERT INTO zrl_.zorologistics.time_travel_demo
SELECT * FROM zrl_.zorologistics.shipments_bronze LIMIT 1000


In [ ]:
%sql
-- Travel: version 1 vs the current state. The row counts must differ.
SELECT 'current'  AS point, count(*) AS rows FROM zrl_.zorologistics.time_travel_demo
UNION ALL
SELECT 'version 1', count(*) FROM zrl_.zorologistics.time_travel_demo VERSION AS OF 1


In [ ]:
%sql
-- The same read with @-syntax and a timestamp pin.
SELECT count(*) AS rows_at_v1 FROM zrl_.zorologistics.time_travel_demo@v1


In [ ]:
%sql
-- The log now has two versions we can travel across.
DESCRIBE HISTORY zrl_.zorologistics.time_travel_demo


## 4. VACUUM, OPTIMIZE ZORDER, liquid clustering

- **VACUUM** deletes data files older than the retention window (default **7 days**). It is irreversible *past that window*, which is why it is a cost-vs-recovery decision. `DRY RUN` previews without deleting.
- **OPTIMIZE** compacts small files; **ZORDER BY** colocates rows on a filter column for data skipping. Both Z-order and Hive partitioning are now superseded by **liquid clustering**.
- **Liquid clustering** (`CLUSTER BY`) auto-organizes data and lets you change keys *without rewriting*. `CLUSTER BY AUTO` lets Databricks adapt keys from workload. Requires DBR 15.4 LTS+.


In [ ]:
%sql
-- VACUUM dry run: preview files eligible for removal (nothing recent will be removed).
VACUUM zrl_.zorologistics.time_travel_demo DRY RUN


In [ ]:
%sql
-- OPTIMIZE compacts files; ZORDER BY colocates rows by the filter column.
OPTIMIZE zrl_.zorologistics.shipments_bronze ZORDER BY (carrier_id)


In [ ]:
%sql
-- Liquid clustering on the bronze table (supersedes partitioning + Z-order).
ALTER TABLE zrl_.zorologistics.shipments_bronze CLUSTER BY (carrier_id, lane_id)


In [ ]:
%sql
-- Confirm the clustering columns took effect (look at clusteringColumns).
DESCRIBE DETAIL zrl_.zorologistics.shipments_bronze


In [ ]:
# Final metric: bronze row count + the number of time-travel versions we can read.
rows = spark.sql("SELECT count(*) FROM zrl_.zorologistics.shipments_bronze").collect()[0][0]
versions = spark.sql("DESCRIBE HISTORY zrl_.zorologistics.time_travel_demo").count()
print("bronze rows:", rows)
print("time-travel versions:", versions)
